<a href="https://colab.research.google.com/github/lotannamoldon/Portfolio-Projects/blob/main/Global_Crop_Yield_Prediction_A_Random_Forest_Regression_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [55]:

# 1. Load the datasets
yield_df = pd.read_csv('yield.csv')
temp = pd.read_csv('temp.csv')
rainfall = pd.read_csv('rainfall.csv')
pesticides = pd.read_csv('pesticides.csv')

# 2. Renaming columns in Temperature to match the others
temp = temp.rename(columns={'country': 'Area', 'year': 'Year'})

# 3. Cleaning Rainfall column names
rainfall.columns = rainfall.columns.str.strip()

# 4. Making sure Rainfall values are numbers
rainfall['average_rain_fall_mm_per_year'] = pd.to_numeric(rainfall['average_rain_fall_mm_per_year'], errors='coerce')

PREPROCESSING PHASE


MERGING THE CSV YIELD AND RAINFALL WITH RELATIONAL COLUMNS

In [56]:
# 1. Select only the necessary columns from the yield data
yield_df = yield_df[['Area', 'Item', 'Year', 'Value']]
# Rename 'Value' to 'yield_hg_per_ha' to be specific
yield_df = yield_df.rename(columns={'Value': 'yield_hg_per_ha'})

# 2. Merge Yield with Rainfall
# We use 'on' to tell pandas which columns to match up
yield_merged = pd.merge(yield_df, rainfall, on=['Year', 'Area'])

# 3. See how many rows we have now
print(f"look after merging Rainfall: {yield_merged.shape}")
yield_merged.head()

look after merging Rainfall: (26105, 5)


,Area,Item,Year,yield_hg_per_ha,average_rain_fall_mm_per_year
0,Afghanistan,Maize,1985,16652,327.0
1,Afghanistan,Maize,1986,16875,327.0
2,Afghanistan,Maize,1987,17020,327.0
3,Afghanistan,Maize,1989,16963,327.0
4,Afghanistan,Maize,1990,17582,327.0


In [57]:
# 1. Selecting only the columns we need
pesticides_subset = pesticides[['Area', 'Year', 'Value']]

# 2. Renaming 'Value' to be more descriptive
pesticides_subset = pesticides_subset.rename(columns={'Value': 'pesticides_tonnes'})

# 3. Merge it into Dataframe
# We do NOT include 'Item' in the 'on' list because
# the pesticide data doesn't specify individual crops.
yield_merged = pd.merge(yield_merged, pesticides_subset, on=['Area', 'Year'])

print(f"New look after adding Pesticides: {yield_merged.shape}")
yield_merged.head()

New look after adding Pesticides: (19356, 6)


,Area,Item,Year,yield_hg_per_ha,average_rain_fall_mm_per_year,pesticides_tonnes
0,Albania,Maize,1990,36613,1485.0,121.0
1,Albania,Maize,1991,29068,1485.0,121.0
2,Albania,Maize,1992,24876,1485.0,121.0
3,Albania,Maize,1993,24185,1485.0,121.0
4,Albania,Maize,1994,25848,1485.0,201.0


In [58]:
# 1. Adding Temperature data
yield_final = pd.merge(yield_merged, temp, on=['Area', 'Year'])

# 2. Final result
print(f"Final dataset shape: {yield_final.shape}")
yield_final.head()

Final dataset shape: (28248, 7)


,Area,Item,Year,yield_hg_per_ha,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,Albania,Maize,1991,29068,1485.0,121.0,15.36
2,Albania,Maize,1992,24876,1485.0,121.0,16.06
3,Albania,Maize,1993,24185,1485.0,121.0,16.05
4,Albania,Maize,1994,25848,1485.0,201.0,16.96


SORTING MISSING DATA

In [59]:
# 1. Calculate the mean (average) of the rainfall column
mean_rainfall = yield_final['average_rain_fall_mm_per_year'].mean()

# 2. Fill the missing values with that mean
yield_final['average_rain_fall_mm_per_year'] = yield_final['average_rain_fall_mm_per_year'].fillna(mean_rainfall)

# 3. Double-check that there are no more missing values
print("Missing values after filling:")
print(yield_final.isnull().sum())

Missing values after filling:
Area                             0
Item                             0
Year                             0
yield_hg_per_ha                  0
average_rain_fall_mm_per_year    0
pesticides_tonnes                0
avg_temp                         0
dtype: int64


In [60]:
# Drop duplicate rows
yield_final = yield_final.drop_duplicates()

# Check the new shape
print(f"Final look after dropping duplicates: {yield_final.shape}")

Final look after dropping duplicates: (25938, 7)


RUNNING AND ENCODING PROCESS

In [61]:
# One-Hot Encoding categorical variables
yield_final = pd.get_dummies(yield_final, columns=['Area', 'Item'], prefix=['Country', 'Crop'])

# Check the new structure
print(f"New shape: {yield_final.shape}")
yield_final.head()

New shape: (25938, 116)


,Year,yield_hg_per_ha,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp,Country_Albania,Country_Algeria,Country_Angola,Country_Argentina,Country_Armenia,...,Crop_Cassava,Crop_Maize,Crop_Plantains and others,Crop_Potatoes,"Crop_Rice, paddy",Crop_Sorghum,Crop_Soybeans,Crop_Sweet potatoes,Crop_Wheat,Crop_Yams
0,1990,36613,1485.0,121.0,16.37,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
1,1991,29068,1485.0,121.0,15.36,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
2,1992,24876,1485.0,121.0,16.06,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3,1993,24185,1485.0,121.0,16.05,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
4,1994,25848,1485.0,201.0,16.96,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False


SPLIT DATA FOR TRAINING

In [62]:

# 1. Define X (features) and y (target)
y = yield_final['yield_hg_per_ha']
X = yield_final.drop(['yield_hg_per_ha'], axis=1)

# 2. Split the data
# test_size=0.2 means 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 20750
Testing samples: 5188


TRAIN MODEL

In [63]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Initialize the model
# We'll start with 100 trees (n_estimators)
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 2. Train the model using our training data
model.fit(X_train, y_train)

# 3. Use the trained model to predict yields for the hidden test set
y_pred = model.predict(X_test)

# 4. Evaluate the accuracy
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R-squared Score: {r2:.4f}")
print(f"Mean Absolute Error: {mae:.2f} hg/ha")

R-squared Score: 0.9868
Mean Absolute Error: 3872.53 hg/ha


In [64]:
import numpy as np

# Get the importance scores
importances = model.feature_importances_
feature_names = X.columns

# Combine them into a list and sort
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Look at the top 10
print(feature_importance_df.head(10))

                           Feature  Importance
108                  Crop_Potatoes    0.373977
105                   Crop_Cassava    0.089391
112            Crop_Sweet potatoes    0.087495
2                pesticides_tonnes    0.074725
46                   Country_India    0.052866
1    average_rain_fall_mm_per_year    0.043653
3                         avg_temp    0.040969
0                             Year    0.033509
114                      Crop_Yams    0.025975
52                   Country_Japan    0.018650


A random forest model was succefully trained to predict crop yield with 98 percent accuracy, Our analysis proves that while the specific type of crop is the strongest indicator of yield, environmental factors and pesticide use provide the critical data the model needs to distinguish between a successful harvest and a poor one.